<a href="https://colab.research.google.com/github/juli0AND/Evaluaci-n-comparativa-de-YOLOv5-y-YOLOv8/blob/main/YOLOv8_NANO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Instalar librerías necesarias
!pip install roboflow
!pip install -q ultralytics roboflow

# 2. Descargar dataset desde Roboflow con la API Key
from roboflow import Roboflow
from PIL import Image
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
rf = Roboflow(api_key="io4o0d2qhkzDHpqNE3Tb")
project = rf.workspace("deteccion-residuos").project("new-classification-waste-yolo_nano-zgtgb")
version = project.version(1)
dataset = version.download("yolov8")


# 3. Verificar el path al archivo data.yaml
yaml_path = dataset.location + "/data.yaml"
print("✅ data.yaml ubicado en:", yaml_path)

# 4. Entrenar YOLOv8
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # También puedes usar yolov8s.pt, yolov8m.pt, etc.

model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project="Roboflow-YOLOv8",
    name="primer_entrenamiento"
)
# Cargar modelo entrenado
model = YOLO("/content/runs/detect/Roboflow-YOLOv8/primer_entrenamiento/weights/best.pt")

# Ejecutar validación para obtener métricas
metrics = model.val()


# Nueva ruta basada en tu entrenamiento
results_img_path = "Roboflow-YOLOv8/primer_entrenamiento/results.png"

if os.path.exists(results_img_path):
    img = Image.open(results_img_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Gráficas del entrenamiento (losses, precision, recall, mAP)")
    plt.show()
else:
    print("❌ No se encontró la gráfica de resultados en:", results_img_path)

conf_matrix_path = "runs/detect/val/confusion_matrix.png"
if os.path.exists(conf_matrix_path):
    img = Image.open(conf_matrix_path)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("No se encontró la matriz de confusión.")+

 # Guardar sin título
    output_path = "matriz_confusion.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0)
    plt.show()

    # Descargar
    from google.colab import files
    files.download(output_path)


# ==============================
# 1. ENTRENAMIENTO
# ==============================
from ultralytics import YOLO
import os
import pandas as pd
import shutil

model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project="Roboflow-YOLOv8",
    name="primer_entrenamiento"
)

# ✅ Ruta correcta del experimento (SIN ERROR)
exp_path = str(results.save_dir)
print("📁 Ruta del experimento:", exp_path)

# ==============================
# 2. EXPORTAR results.csv
# ==============================

results_csv_path = os.path.join(exp_path, "results.csv")

if os.path.exists(results_csv_path):
    print("✅ results.csv encontrado")

    # Copia con nombre más claro
    new_csv_path = os.path.join(exp_path, "training_metrics_full.csv")
    shutil.copy(results_csv_path, new_csv_path)

    print("📄 CSV exportado como:", new_csv_path)

    # Mostrar preview
    df = pd.read_csv(new_csv_path)
    print("\n🔍 Vista previa:")
    print(df.head())

else:
    print("❌ No se encontró results.csv")

# ==============================
# 3. VALIDACIÓN FINAL
# ==============================

# Cargar mejor modelo automáticamente
best_model_path = os.path.join(exp_path, "weights", "best.pt")
model = YOLO(best_model_path)

metrics = model.val()

# ==============================
# 4. EXPORTAR MÉTRICAS FINALES
# ==============================

final_metrics = {
    "precision": metrics.box.mp,
    "recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP50-95": metrics.box.map
}

final_df = pd.DataFrame([final_metrics])
final_metrics_path = os.path.join(exp_path, "final_metrics.csv")

final_df.to_csv(final_metrics_path, index=False)

print("\n✅ Métricas finales exportadas en:", final_metrics_path)
print(final_df)

# ==============================
# 5. DESCARGA (COLAB)
# ==============================

try:
    from google.colab import files

    files.download(new_csv_path)
    files.download(final_metrics_path)


except:
    print("📥 Descarga automática solo disponible en Google Colab")

# ==========================================================
# EXPORTAR CSV
# ==========================================================
csv_name = f"{MODEL_NAME}_resumen.csv"

df.to_csv(csv_name, index=False)

# ==========================================================
# EXPORTAR IMÁGENES EN ALTA RESOLUCIÓN (300 DPI)
# ==========================================================
from PIL import Image as PILImage

# RESULTS.PNG
img = PILImage.open(results_path)
img.save(
    "results_yolov8n_300dpi.png",
    dpi=(300,300)
)

# CONFUSION MATRIX
img2 = PILImage.open(confusion_path)
img2.save(
    "confusion_matrix_yolov8n_300dpi.png",
    dpi=(300,300)
)

# ==========================================================
# DESCARGAR ARCHIVOS IMPORTANTES
# ==========================================================
files.download(csv_name)

files.download("results_yolov8n_300dpi.png")

files.download("confusion_matrix_yolov8n_300dpi.png")

files.download(WEIGHTS)

files.download(
    "runs/detect/yolov8_residuos/results.csv"
)